# 📈 โครงงานวิจัย: การจัดสรรพอร์ตการลงทุนด้วย NSGA-II
**(Multi-Objective Portfolio Optimization using NSGA-II under Real-world Constraints)**

สมุดบันทึก (Jupyter Notebook) เล่มนี้แสดงกระบวนการทำงานแบบ End-to-End ของการแก้ปัญหาการจัดพอร์ตการลงทุนที่ซับซ้อนระดับ NP-Hard โดยใช้เทคโนโลยี AI (Evolutionary Algorithm) ผสมผสานกับทฤษฎีการเงิน

## 1. ทฤษฎีและสมการคณิตศาสตร์ (Mathematical Formulation)

เราอ้างอิงทฤษฎี **Modern Portfolio Theory (MPT)** ของ Harry Markowitz โดยปรับสมการให้เป็น Multi-objective:

**Objective 1: Minimize Risk (จำกัดความเสี่ยงให้น้อยที่สุด)**
$$ \min f_1(w) = \sqrt{w^T \Sigma w} + \lambda \cdot Penalty_{sector} $$
*   $w$: เวกเตอร์น้ำหนักการลงทุน
*   $\Sigma$: เมทริกซ์ความแปรปรวนร่วม (Covariance Matrix)

**Objective 2: Maximize Return (เพิ่มผลกำไรสูงสุดหลังหักค่าธรรมเนียม)**
$$ \max f_2(w) = w^T \mu - (Turnover \times c) - \lambda \cdot Penalty_{sector} $$
*   $\mu$: เวกเตอร์ผลตอบแทนคาดหวัง
*   $Turnover$: อัตราการหมุนเวียนพอร์ต
*   $c$: ค่าคอมมิชชัน (0.20%)

**ข้อจำกัดในโลกความเป็นจริง (Real-world Constraints):**
1. **Budget:** ผลรวมเงินลงทุนต้องเท่ากับ 100% ($\sum w_i = 1$)
2. **Cardinality:** เลือกลงทุนหุ้นเพียง $K = 10$ ตัว จากทั้งหมด $N = 30$ ตัว
3. **Boundary:** ถือหุ้นต่อตัวขั้นต่ำ 5% และสูงสุดไม่เกิน 25% ($0.05 \le w_i \le 0.25$)
4. **Sector Limit:** ห้ามถือหุ้นในอุตสาหกรรมเดียวกันเกิน 40%

## 2. การตั้งค่าระบบ (Initialization & Configuration)
เรานำเข้าเครื่องมือต่างๆ และตั้งค่า Seed เพื่อให้การประมวลผลสามารถทำซ้ำได้ (Reproducibility)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pymoo.core.problem import ElementwiseProblem
from pymoo.core.repair import Repair
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.operators.sampling.rnd import FloatRandomSampling

# นำเข้าโมดูลย่อยที่เราพัฒนาไว้ในโฟลเดอร์ src/
from src.algorithms.optimizer import PortfolioOptimizer
from src.utils.visualization import (
    plot_convergence_curve,
    plot_pareto_front,
    plot_selected_weights,
    plot_cumulative_returns
)

# การันตี Reproducibility ให้กับโมเดล
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 3. การดึงข้อมูลทางการเงิน (Data Acquisition & Train-Test Split)
เราใช้คลาส `PortfolioOptimizer` ซึ่งจะทำการดึงข้อมูลจาก Yahoo Finance ย้อนหลัง 10 ปีโดยอัตโนมัติ และคำนวณผลตอบแทนรายวัน

**การป้องกัน Look-ahead bias:**
เราตัดข้อมูล 252 วันสุดท้าย (1 ปี) แยกออกไปเป็น Test Set โมเดล AI จะไม่มีทางเห็นข้อมูลนี้ตอนฝึกสอน (Train)

In [ ]:
# สร้างออบเจกต์และดาวน์โหลดข้อมูล
opt = PortfolioOptimizer(experiment_name="NSGA2_Sector_Constrained")
opt.fetch_and_prepare_data()

print(f"📌 จำนวนหุ้นใน Universe: {opt.n_assets} ตัว")
print(f"📌 ข้อมูลใช้ฝึกสอน (Train Set): {opt.train_returns.shape[0]} วันทำการ")
print(f"📌 ข้อมูลใช้ทดสอบจริง (Test Set): {opt.test_returns.shape[0]} วันทำการ")

## 4. กลไกแกนกลาง: 2N Encoding และ Cardinality Repair (โชว์ซอร์สโค้ดเต็ม)

การบังคับให้ AI เลือกหุ้น 10 ตัวเป๊ะๆ เป็นปัญหา NP-Hard การใช้ตัวแปร 0/1 (Binary) ปกติมักจะทำให้ AI หลงทาง เราจึงแก้ปัญหาด้วยทฤษฎี **2N Encoding**:
เราให้โครโมโซมมีความยาว 60 ยีน (2 * 30)
- **30 ยีนแรก:** คือค่าน้ำหนักการลงทุน (Weight Genes)
- **30 ยีนหลัง:** คือคะแนนความน่าเลือก (Selection Genes)

เมื่อโมเดลผสมพันธุ์เสร็จ เราจะใช้คลาส `Repair` ตัดหุ้นที่คะแนนไม่ติด Top 10 ทิ้ง และเกลี่ยน้ำหนักใหม่ให้อยู่ในกรอบ 5%-25%

In [ ]:
class PortfolioRepair(Repair):
    def __init__(self, optimizer):
        super().__init__()
        self.opt = optimizer

    def _do(self, problem, X, **kwargs):
        n = self.opt.n_assets
        for i in range(len(X)):
            # แกะกล่อง 2N Encoding
            raw_weights = X[i, :n]
            selection_scores = X[i, n:]
            
            # ส่งเข้ากระบวนการคัดกรอง Top K (Cardinality) และเกลี่ยน้ำหนักให้อยู่ในกรอบ (Boundary)
            weights = self.opt.project_weights_with_cardinality(raw_weights, selection_scores)
            
            # จำลองการเทรดจริงด้วยการปัดเศษเป็น 1% (Transaction Lots)
            weights = np.round(weights, 2)
            if np.sum(weights) > 0:
                weights = weights / np.sum(weights) # Budget Constraint (sum = 1)
                
            X[i, :n] = weights
        return X

## 5. กลไกแกนกลาง: เป้าหมายการลงทุน และระบบลงโทษ (Problem Setup)

คลาสนี้คือตัวให้คะแนนโครโมโซม (Fitness Evaluation) เราเขียนโค้ดแบบ **Vectorization (คูณเมทริกซ์)** เพื่อความเร็ว
หากหุ้นอุตสาหกรรมเดียวกัน (Sector) กระจุกตัวเกิน 40% เราจะสร้างบทลงโทษ `sector_penalty` มหาศาลไปทำลายคะแนนของมัน (Death Penalty)

In [ ]:
class PortfolioProblem(ElementwiseProblem):
    def __init__(self, optimizer):
        self.opt = optimizer
        n = self.opt.n_assets
        super().__init__(n_var=2*n, n_obj=2, n_constr=0, xl=0, xu=1)

    def _evaluate(self, x, out, *args, **kwargs):
        weights = x[:self.opt.n_assets]
        
        # 1. Expected Return = w^T * mu
        ann_return = float(np.dot(weights, self.opt.mu_train.values))
        
        # 2. Risk = sqrt(w^T * Covariance * w)
        variance = float(weights.T @ self.opt.cov_train.values @ weights)
        ann_vol = float(np.sqrt(max(variance, 1e-12)))
        
        # 3. Transaction Costs Penalty (หัก 0.20% ของอัตราหมุนเวียน)
        eq_weights = self.opt.build_equal_weight()
        turnover = float(0.5 * np.sum(np.abs(weights - eq_weights)))
        net_return = ann_return - (turnover * 0.002)
        
        # 4. Sector Concentration Penalty
        sector_penalty = 0.0
        if self.opt.max_sector_weight is not None:
            exposures = self.opt.sector_exposures(weights)
            sector_penalty = max(0.0, max(exposures.values()) - self.opt.max_sector_weight)
        
        # 5. Output (Pymoo บังคับ Minimize เราจึงส่ง return แบบติดลบเพื่อให้มัน Maximize)
        out["F"] = [
            ann_vol + (sector_penalty * 50.0),
            -net_return + (sector_penalty * 50.0)
        ]

## 6. รันอัลกอริทึม NSGA-II (Evolutionary Process)
เราสั่งรันอัลกอริทึมทั้งหมดแบบอัตโนมัติ โดยเรียกใช้โมดูลออพติไมเซอร์จากโฟลเดอร์ `src/` ซึ่งมีการประกอบร่างคลาสข้างต้นเข้าด้วยกันเรียบร้อยแล้ว

In [ ]:
# การสั่งรันคำสั่งเดียว (Encapsulation) จะครอบคลุมการรัน NSGA-II 300 รุ่น
# และคืนผลลัพธ์มาในรูปแบบของแพ็กเกจ Artifacts
artifacts = opt.run()

print(f"\n🔥 ได้จุดที่ดีที่สุด (Max Sharpe Ratio): {artifacts.objective:.4f} ในฝั่ง In-Sample")

## 7. วิเคราะห์ผลการทดลอง (Experiment Analysis)
### 7.1 การลู่เข้าหาคำตอบและประสิทธิภาพอัลกอริทึม (Convergence & Pareto Front)
กราฟเหล่านี้พิสูจน์ว่า AI ของเราเกิดการเรียนรู้และพัฒนาพอร์ตให้ดีขึ้นในแต่ละรุ่น (Generation)

In [ ]:
# กราฟความก้าวหน้าของความฉลาด (Convergence)
plot_convergence_curve(artifacts.convergence_history_df, artifacts.experiment_name, "outputs/nb_convergence.png")

# กราฟเส้นความสมดุลระหว่างความเสี่ยงและผลตอบแทน (Pareto Front)
plot_pareto_front(opt.pareto_df, artifacts.experiment_name, "outputs/nb_pareto.png")

from IPython.display import Image, display
display(Image("outputs/nb_convergence.png"))
display(Image("outputs/nb_pareto.png"))

### 7.2 ตรวจสอบวินัยการลงทุน (Constraint Audit)
เรามาดูกันว่า AI เคารพกฎข้อบังคับเรื่อง Cardinality และ Sector Limit หรือไม่

In [ ]:
plot_selected_weights(artifacts.weights_table, opt.sector_colors, artifacts.experiment_name, "outputs/nb_weights.png")
display(Image("outputs/nb_weights.png"))

print("✅ สรุปน้ำหนักรายอุตสาหกรรม (จะต้องไม่มีหมวดไหนเกิน 40%):")
display(artifacts.sector_table.head(5))

### 7.3 จำลองการลงทุนในโลกจริง 1 ปีล่าสุด (Out-of-sample Backtesting)
เอาเงิน 10,000 ดอลลาร์ไปจำลองเทรดจริงในช่วง 252 วันสุดท้าย (AI ไม่เคยเห็นข้อมูลส่วนนี้มาก่อน)

In [ ]:
plot_cumulative_returns(artifacts.nsga2_curve, artifacts.equal_curve, artifacts.benchmark_curve, artifacts.experiment_name, opt.benchmark_ticker, "outputs/nb_backtest.png")
display(Image("outputs/nb_backtest.png"))

print("\n💵 สรุปสถิติทางการเงิน (Out-of-sample Financial Metrics):")
display(artifacts.metrics_summary)

##  สรุปผลการวิจัย (Conclusion)
โมเดล **NSGA-II** ทำงานผสานกับสมการ **Mean-Variance** ได้อย่างยอดเยี่ยม:
1. **ระบบสามารถทะลุข้อจำกัดได้จริง:** ด้วยนวัตกรรม 2N Encoding และ Penalty Function ทำให้ AI สามารถจัดพอร์ตที่เลือกหุ้น 10 ตัว และคุมความเสี่ยงราย Sector ไม่เกิน 40% ได้อย่างสมบูรณ์แบบ
2. **เอาชนะความเสี่ยงวิกฤต:** ค่า Maximum Drawdown ในฝั่ง Out-of-sample ต่ำกว่าพอร์ตตลาด (S&P 500) และทำผลกำไรระยะยาวได้เสถียร พิสูจน์ให้เห็นถึงประสิทธิภาพของการ Optimize ด้วย AI